# Chapter 8: Avoiding Hallucinations

**Technique:** Grounding to reduce hallucinations

**Contents**
* [Lesson: Why Models Confabulate and How to Stop It](#lesson)
* [Exercises](#exercises)
* Example Playground

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Why Models Confabulate and How to Stop It

Language models are trained to produce fluent, coherent text. When they don't know something they will often produce a confident sounding answer anyway, a behaviour called **confabulation** or hallucination. For production code this is a reliability problem: a model that quietly invents an API signature, a package name, or a configuration value will silently introduce bugs.

Two complementary defences work well in practice.

### Defence 1: Give Claude an explicit "out"

By default, Claude tries to be helpful. If you ask "How do I use `@acme/quantum-router`?" it will try to answer. You can short circuit this by telling Claude it is allowed, even expected, to say it doesn't know:

```js
const PROMPT = `You are a senior JavaScript engineer. Answer only from what you know with confidence.
If you are not sure or if the package / API does not exist, say so explicitly. Do not guess.

Does the npm package @acme/quantum-router expose a \`connect()\` method? If so, what does it do?`;
```

Adding "if you are not sure, say so" shifts Claude's default from *generate a plausible answer* to *flag uncertainty first*.

### Defence 2: Ground answers in a provided document and require quotes

For customer support bots, RAG pipelines, and document Q&A you often have the source of truth at hand. Pass it to Claude explicitly and instruct it to:

1. Quote the relevant passage verbatim before answering.
2. Decline, do not guess, if the answer is not present.

```js
const PROMPT = `Answer the question below using ONLY the text in <document>.
Before your answer, quote the relevant lines verbatim inside <quote> tags.
If the answer is not present in the document, say "Not specified in the document", and do not guess.

Question: What is the maximum payload size per request?
<document>
${DOCUMENT}
</document>`;
```

This forces a traceable, auditable answer and makes the absence of information explicit rather than silently papered over.

### Both defences together

Use the "out" whenever the model's training data might be wrong or stale. Use document grounding whenever you own the authoritative source. In a RAG system you typically apply both: retrieved chunks go into the document, and the model is told to decline if the chunk doesn't contain the answer.

In [ ]:
// Demonstrate: asking about a fabricated package WITHOUT and WITH an explicit "out"
const FAKE_PKG = "@acme/quantum-router";

const WITHOUT_OUT = `What is the \`connect()\` method signature in ${FAKE_PKG}?`;

const WITH_OUT = `You are a senior JavaScript engineer.
If you are not confident about a package or API, including whether it exists at all, say so clearly.
Do not guess or invent details.

What is the \`connect()\` method signature in ${FAKE_PKG}?`;

console.log("=== Without explicit out ===");
console.log(await getCompletion(WITHOUT_OUT));

console.log("\n=== With explicit out ===");
console.log(await getCompletion(WITH_OUT));

## Exercises

### Exercise 8.1: Give Claude an "out" for an unknown package

**Scenario**: a teammate's PR references the npm package `@acme/quantum-router`. You've never heard of it and want to check whether its `RouteTable` class has a `prefixMatch()` method before you approve the PR. You ask Claude, but this package does not exist. Without an explicit "out", Claude may fabricate a convincing sounding answer.

**Task**: write a prompt that asks Claude about `@acme/quantum-router` and its `RouteTable.prefixMatch()` method, but explicitly gives Claude permission, and encouragement, to admit it doesn't know.

**Success criteria**: Claude's response admits uncertainty (contains a phrase like "don't know", "not aware", "no information", etc.) rather than inventing package details.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const PROMPT = "You are a senior JavaScript engineer. If you are not certain that a package or API exists, say so plainly and do NOT guess or invent details. Does the npm package @acme/quantum-router exist, and does its RouteTable class have a prefixMatch() method? If you have no information about it, say so.";

const response = await getCompletion(PROMPT);
const admits = includesAny(response, ["don't", "do not", "not aware", "unable", "no information", "couldn't find", "could not find", "not familiar"]);
const gradeExercise = (text) => admits;
grade(response, gradeExercise(response));

In [ ]:
import { exercise_8_1_hint } from "../hints.js";
console.log(exercise_8_1_hint);

### Exercise 8.2: Answer only from a provided document

**Scenario**: your team ships an internal `DataBridge` service and maintains its API reference as a Markdown document. You're building a support bot that should answer questions strictly from this document; it must never guess. A user asks: *"What is the API request rate limit?"* That information is **not** in the document, so the bot should decline rather than invent a number.

**Task**: write a prompt that passes the document below to Claude, asks the rate limit question, and instructs Claude to:
1. Quote the relevant passage first (inside `<quote>` tags).
2. Answer only from the document.
3. Decline explicitly if the answer is absent.

**Success criteria**: Claude's response indicates the answer is not present (contains a phrase like "not mentioned", "does not", "no information", "not specified", etc.).

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const DOCUMENT = `# DataBridge Service — API Reference v2.4

## Overview

DataBridge is an internal HTTP/JSON service that moves records between upstream data producers and downstream analytics consumers. Every request is authenticated with a short-lived bearer token obtained from the corporate identity provider. Tokens expire after 60 minutes; your client must refresh them before expiry using the \`/auth/token\` endpoint.

## Endpoints

### POST /v2/ingest

Ingest one or more records into a named dataset. The request body must be a JSON object with the following fields:

- \`dataset\` (string, required): the target dataset identifier, e.g. \`"events.clickstream"\`.
- \`records\` (array, required): an array of record objects. Each record must include a \`timestamp\` (ISO-8601 string) and a \`payload\` (arbitrary JSON object). Maximum 500 records per request; batches exceeding this limit are rejected with HTTP 413.
- \`dedup_key\` (string, optional): when present, DataBridge discards duplicate records that share the same \`dedup_key\` within a 24-hour rolling window.

**Response**: HTTP 202 Accepted. Body: \`{ "accepted": <count>, "rejected": <count>, "batch_id": "<uuid>" }\`.

### GET /v2/datasets

Returns a paginated list of datasets the caller has read access to. Query parameters:

- \`page\` (integer, default 1): page number, 1-indexed.
- \`page_size\` (integer, default 20, max 100): number of datasets per page.

**Response**: HTTP 200. Body: \`{ "datasets": [...], "total": <count>, "page": <n>, "page_size": <n> }\`.

### DELETE /v2/datasets/{id}

Permanently deletes a dataset and all of its records. This action is irreversible. The caller must hold the \`datasets:delete\` scope on their token. Deletion is asynchronous; the dataset transitions to \`PENDING_DELETE\` status immediately and is fully removed within 72 hours.

## Error Codes

| HTTP Status | Code | Meaning |
|-------------|------|---------|
| 400 | \`INVALID_PAYLOAD\` | Request body failed schema validation. |
| 401 | \`TOKEN_EXPIRED\` | Bearer token has expired; obtain a new one. |
| 403 | \`INSUFFICIENT_SCOPE\` | Token lacks the required permission scope. |
| 413 | \`BATCH_TOO_LARGE\` | Batch exceeds the 500-record limit per request. |
| 503 | \`SERVICE_UNAVAILABLE\` | DataBridge is temporarily unavailable; retry with exponential backoff. |

## Authentication

All endpoints require an \`Authorization: Bearer <token>\` header. Tokens are issued by calling \`POST /auth/token\` with your service-account credentials. Set \`Content-Type: application/json\` and include \`{ "client_id": "<id>", "client_secret": "<secret>", "scope": "<space-separated scopes>" }\`. The response includes \`access_token\`, \`token_type\` (\`"Bearer"\`), and \`expires_in\` (seconds).

## Changelog

**v2.4** — Added \`dedup_key\` support on \`/v2/ingest\`. Deduplication window extended from 1 hour to 24 hours.
**v2.3** — Introduced \`DELETE /v2/datasets/{id}\`. Deletion is now asynchronous with a \`PENDING_DELETE\` transition state.
**v2.2** — \`page_size\` cap on \`GET /v2/datasets\` increased from 50 to 100.
**v2.1** — Bearer token expiry reduced from 120 minutes to 60 minutes to align with corporate security policy.`;

const PROMPT = "Answer the question using ONLY the text inside the <document> tags below. First quote the relevant lines verbatim inside <quote></quote> tags. If the answer is not present in the document, say it is not specified in the document and do not guess. Question: What is the API request rate limit?\n<document>\n" + DOCUMENT + "\n</document>";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAny(text, ["not mentioned", "does not", "doesn't", "no information", "not in the document", "cannot find", "can't find", "not specified"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_8_2_hint } from "../hints.js";
console.log(exercise_8_2_hint);

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **No explicit "out"**: asking "Does package X have method Y?" without permission to say "I don't know" invites confabulation. The model defaults to helpfulness, and if it can generate a plausible sounding answer it will. Always add a line like "If you are not sure, say so explicitly."
* **Asking without grounding**: for document Q&A, passing the document without the instruction "answer only from the document" still lets Claude supplement with its training data. The grounding instruction must be explicit: *only* from the document, not from general knowledge.
* **Soft declines that still leak invented details**: a response like "I'm not 100% sure, but the rate limit is probably 1 000 req/s" passes a soft admission test but still fabricates information. Your grader and production checks should look for fabricated specifics, not just a hedging phrase.

**Stretch challenge**

Extend Exercise 8.2: add a second question whose answer *is* in the document (e.g., "What is the maximum number of records per ingest request?") and require Claude to quote the exact line before answering. Verify that (a) it quotes correctly and (b) the answer matches the document. Now run both questions in one prompt and observe whether grounding holds for both.

**Reflect**: how does this map to RAG systems you might build? In a retrieval augmented pipeline, the retrieval step provides the "document", but retrieved chunks may be incomplete or misranked. What happens when the relevant chunk is missing from the retrieval results? Combining explicit "out" instructions with strict grounding means the model surfaces the gap rather than silently filling it with invented content, which is exactly the failure mode you want to catch in production.

## Example Playground

Use the cells below to experiment freely. Try changing the question to one that *is* answered in the DataBridge document and observe how Claude's quoting behaviour changes. Or try asking about a different fabricated package and varying how strongly you give Claude permission to decline.

In [ ]:
// Experiment: ask a question the document DOES answer, observe quoting behaviour
const SAMPLE_DOC = `DataBridge API v2.4 — Quick Reference

POST /v2/ingest accepts up to 500 records per request (HTTP 413 if exceeded).
Tokens expire after 60 minutes; refresh via POST /auth/token.`;

const QUESTION = "What happens if I send more than 500 records in one request?";

const PROMPT = `Answer the question below using ONLY the text in <document>.
Before your answer, quote the relevant passage verbatim inside <quote> tags.
If the answer is not in the document, respond with "Not specified in the document."

Question: ${QUESTION}
<document>
${SAMPLE_DOC}
</document>`;

console.log(await getCompletion(PROMPT));